# Entrenar el checkpoint NLLB-200 + LoRA (F. Prado) para retrotraducción

Este checkpoint es lo que necesita `4_aumento_datos/retrotraduccion.py` para traducir
las paráfrasis en español al shiwilu (paso 2 de la técnica).

Antes de correr: `Entorno de ejecución` -> `Cambiar tipo de entorno de ejecución` -> **GPU**.

**IMPORTANTE:** cada celda de abajo empieza con `%cd /content/...` explícito, a
propósito — Colab a veces reinicia el entorno solo (por ejemplo tras instalar
`torch`), y eso borra en qué carpeta estabas parada. Si ves un error de
`ModuleNotFoundError` o `No such file or directory`, casi siempre es por eso:
vuelve a correr **desde la celda 1** en adelante, no solo la que falló.

## 1. Montar Google Drive (para guardar el checkpoint antes de que se cierre la sesión)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clonar tu repo y el de F. Prado

In [ ]:
GITHUB_USUARIO = "TU-USUARIO-AQUI"  # <-- cambia esto

%cd /content
!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd /content/shiwilu-tesis
!git clone https://github.com/fapi19/Tesis_Spa-Jeb.git 4_aumento_datos/tesis_spa_jeb

## 3. Instalar dependencias (las que F. Prado ya fijó y probó)

In [ ]:
%cd /content/shiwilu-tesis/4_aumento_datos/tesis_spa_jeb
# Creamos un entorno virtual AISLADO del Python de Colab. Colab trae mas de
# 100 paquetes preinstalados (jax, tensorflow, opencv, wandb, etc.) con sus
# propias versiones de numpy/protobuf, y al instalar los requisitos de F.
# Prado encima de eso se generaban conflictos que dejaban TODO el paquete
# `transformers` roto por dentro. En un entorno aislado, ninguno de esos
# paquetes preinstalados esta presente, asi que no hay con que chocar.
#
# Usamos `virtualenv` (no el modulo estandar `venv`) porque `venv` falla en
# Colab al intentar instalar pip dentro del entorno nuevo (paso `ensurepip`
# roto en la distribucion de Ubuntu que usa Colab). `virtualenv` trae su
# propio pip empaquetado y no depende de ese paso.
!pip install -q virtualenv
!rm -rf /content/nmt_venv
!virtualenv -q /content/nmt_venv
!/content/nmt_venv/bin/pip install -q -r requirements/nmt.txt
!/content/nmt_venv/bin/pip install -q sentence-transformers
!/content/nmt_venv/bin/python -c "import torch; print('GPU disponible:', torch.cuda.is_available())"
print('Entorno listo en /content/nmt_venv. De aqui en adelante, las celdas usan /content/nmt_venv/bin/python en vez de python normal.')

## 4. Entrenar la configuración campeona (v2.1b LoRA+)

Esto puede tardar 30 min - 2 horas segun la GPU que te toque. Nota el `%cd`
explícito al inicio: si el entorno se reinició en la celda 3, esto lo
corrige solo.

In [ ]:
%cd /content/shiwilu-tesis/4_aumento_datos/tesis_spa_jeb
!/content/nmt_venv/bin/python -m scripts.nmt.30_train_lora \
    --variant xl \
    --rank 32 \
    --alpha 64 \
    --loraplus-lr-ratio 16 \
    --output-dir models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl

## 5. Guardar el checkpoint en Drive (¡no te saltes este paso!)

In [ ]:
%cd /content/shiwilu-tesis/4_aumento_datos/tesis_spa_jeb
!mkdir -p /content/drive/MyDrive/shiwilu_checkpoint
!cp -r models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl /content/drive/MyDrive/shiwilu_checkpoint/

## 6. Evaluar (deberia dar chrF++ promedio cercano a 44.99)

In [ ]:
%cd /content/shiwilu-tesis/4_aumento_datos/tesis_spa_jeb
!/content/nmt_venv/bin/python -m scripts.nmt.40_evaluate --checkpoint models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl --split test

## 7. Correr la retrotraducción con tu script

Ya con el checkpoint entrenado, corre la técnica completa (parafraseo + traducción a shiwilu + filtros).

In [ ]:
%cd /content/shiwilu-tesis
!/content/nmt_venv/bin/python 4_aumento_datos/retrotraduccion.py \
    --checkpoint 4_aumento_datos/tesis_spa_jeb/models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl

## 8. Descargar el resultado a tu computadora

Descarga solo el CSV generado (no el checkpoint completo, pesa varios GB).

In [ ]:
%cd /content/shiwilu-tesis
from google.colab import files
files.download('4_aumento_datos/salidas/retrotraduccion.csv')